# Locally weighted regression (LWR)

_Machine Learning — more_

**Fit a fresh line at every query point, trusting nearby data most.**

Plain linear regression fits one line to all the data. That fails for curvy patterns.

     Locally weighted regression fits a fresh little line for each point you ask about.

---

This notebook is a practice scaffold for the **Locally weighted regression (LWR)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

Locally weighted regression refits a fresh weighted line for every query point, trusting nearby data most. We build it in three steps: (1) make some curvy data, (2) write the per-query weighted fit, (3) see how the bandwidth `tau` changes the result.

### Step 1 — Make a curvy dataset

We build a deterministic wiggly relationship, `y = sin(x) + 0.25x`, plus a little noise. A single straight line can never follow this shape — which is exactly the situation LWR is built for. `X` is the design matrix with a leading column of 1s for the intercept.

In [ ]:
import numpy as np
rng = np.random.RandomState(0)

# Deterministic curvy data: y = sin(x) + 0.25x + noise.
x = np.linspace(-5, 5, 40)
y = np.sin(x) + 0.25 * x + 0.15 * rng.randn(40)

X = np.c_[np.ones_like(x), x]        # design matrix with intercept column

### Step 2 — Fit a weighted line at one query point

For a query point `xq`, every training point gets a Gaussian weight that decays with its distance from `xq`: close points count fully, far points barely at all. We then solve the **weighted normal equations** `(Xᵀ W X) theta = Xᵀ W y` for that local line and read off its prediction at `xq`. The bandwidth `tau` sets how quickly the weights fall off.

In [ ]:
def lwr_predict(xq, tau):
    # Gaussian weights: nearby points count most, far points fade out.
    weights = np.exp(-((x - xq) ** 2) / (2 * tau ** 2))
    W = np.diag(weights)

    # Weighted normal equations: (X^T W X) theta = X^T W y
    A = X.T @ W @ X
    b = X.T @ W @ y
    theta = np.linalg.solve(A, b)

    # Evaluate the local line at the query point.
    query_row = np.array([1.0, xq])
    return theta @ query_row

### Step 3 — See how bandwidth `tau` changes the fit

We predict at three query points for three bandwidths. A **small** `tau` makes the weights very local, so the fit hugs the data and looks wiggly; a **large** `tau` weights almost everything equally, so the fit collapses back toward a single straight line.

In [ ]:
for tau in [0.3, 0.8, 3.0]:
    query_points = [-3.0, 0.0, 3.0]
    preds = [lwr_predict(xq, tau) for xq in query_points]
    print("tau=%.1f  preds@(-3,0,3) = [%.3f, %.3f, %.3f]" %
          (tau, preds[0], preds[1], preds[2]))

print("small tau hugs the data (wiggly); large tau ~ one straight line")

## Visualize the data & results

_Can we fit a real noisy relationship without one global line?_

Now we run LWR on **real** data — diabetes patients’ BMI versus disease progression — and draw the locally-weighted curve over the scatter. We build it in three steps: (1) load the real data and define the weighted fit, (2) evaluate the fit across a grid of BMI values, (3) plot the patients and the curve.

### Step 1 — Load real data and define the weighted fit

We use 442 diabetes patients, taking the standardized BMI as the single feature and one-year disease progression as the target. The `lwr_predict` helper is the same weighted least-squares fit as before, with a tiny `1e-6` ridge added to the matrix so the solve stays stable where data is sparse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# REAL data: 442 diabetes patients, feature 2 = standardized BMI.
db = load_diabetes()
bmi = db.data[:, 2]
prog = db.target                         # disease progression after one year
X = np.c_[np.ones_like(bmi), bmi]        # design matrix with intercept

def lwr_predict(xq, tau=0.04):
    weights = np.exp(-((bmi - xq) ** 2) / (2 * tau ** 2))    # Gaussian weights near xq
    W = np.diag(weights)
    A = X.T @ W @ X + 1e-6 * np.eye(2)                       # ridge keeps the solve stable
    b = X.T @ W @ prog
    theta = np.linalg.solve(A, b)                            # weighted normal equations
    query_row = np.array([1.0, xq])
    return float(theta @ query_row)

### Step 2 — Evaluate the fit across the BMI range

We pick 60 patients at random just to keep the scatter readable, then sweep a grid of BMI values from the smallest to the largest, calling `lwr_predict` at each. Every grid point gets its *own* freshly-weighted line — that is what lets the overall curve bend.

In [ ]:
# Show 60 patients to keep the scatter readable.
rng = np.random.RandomState(0)
idx = np.sort(rng.choice(len(bmi), 60, replace=False))

# Sweep BMI from low to high, refitting a weighted line at each grid point.
grid = np.linspace(bmi.min(), bmi.max(), 50)
fit = [lwr_predict(q) for q in grid]

### Step 3 — Plot the patients and the local fit

We scatter the sampled patients and overlay the locally-weighted curve. The curve bends to follow the cloud of points instead of forcing a single global straight line through them — the whole point of LWR.

In [ ]:
plt.scatter(bmi[idx], prog[idx], color='#4ea1ff', label='442 real patients (60 shown)')
plt.plot(grid, fit, color='#ffb454', label='LWR fit (refit a weighted line at every BMI)')
plt.xlabel('body mass index (standardized)')
plt.ylabel('disease progression after one year')
plt.title('Diabetes: BMI vs disease progression with a locally-weighted fit')
plt.legend()
plt.show()